# Sentence-BERT Semantic Search vs. TF-IDF / BM25

A from-scratch walkthrough of building a semantic search engine with **Sentence-BERT (SBERT)**
embeddings, comparing it against the classic lexical baselines **TF-IDF** and **BM25**.

## What you'll learn

1. **Why raw BERT is bad at sentence similarity**, and what SBERT changes
2. How **bi-encoders** turn sentences into fixed-size vectors that you can compare with cosine similarity
3. How to build three search engines over the same corpus: **TF-IDF**, **BM25**, and **SBERT**
4. How to **benchmark** them on retrieval quality (Precision@k, MRR) and on **latency**
5. When each method wins, and why a real production system usually **combines** them

---

## 1. The core problem: turning text into comparable vectors

Search means: given a query, find the most relevant documents in a corpus. To do that *computationally*,
both the query and every document must become **vectors** that live in the same space, so we can measure
"closeness" between them (almost always with **cosine similarity**).

There are two fundamentally different ways to build those vectors:

| Approach | How it represents text | What it captures |
|---|---|---|
| **Lexical (TF-IDF, BM25)** | Counts of *exact words/tokens* | Word overlap — "did the same words appear?" |
| **Semantic (SBERT)** | Dense vector from a neural network | Meaning — "do these sentences mean the same thing, even with different words?" |

Lexical methods will completely miss that *"How do I reset my password?"* and *"I forgot my login credentials"*
are asking the same thing, because they share almost no words. Semantic embeddings are built precisely to catch this.

---

## 2. Why not just use raw BERT?

BERT produces a vector **for every token**, not one vector per sentence. To get a single sentence vector,
the naive approach is to average all token vectors (mean pooling) or just take the `[CLS]` token's vector.

This sounds reasonable, but Reimers & Gurevych (the SBERT paper, EMNLP 2019) showed it works **badly**:
raw BERT/RoBERTa token-averaging produces sentence embeddings that are often *worse* than older, simpler
methods like averaged GloVe vectors, when measured on Semantic Textual Similarity (STS) benchmarks.

**Why does this happen?** BERT was pretrained with masked language modeling and next-sentence prediction —
objectives about *predicting tokens*, not about making "the vector of sentence A" and "the vector of sentence B"
sit close together in space *only when A and B mean similar things*. Nothing in BERT's pretraining ever
pushes whole-sentence representations into a metric space where cosine similarity is meaningful. The geometry
of that embedding space is essentially an accident of the pretraining objective, not a designed property.

There's also a **practical/speed** problem with using BERT for search directly. The most accurate way to
compare two sentences with BERT is to feed BOTH into the model together (as `[CLS] sentence A [SEP] sentence B [SEP]`)
and read off a similarity score from the output — this is called a **cross-encoder**. It's accurate because
the model can attend between the two sentences. But it means that to find the best match for 1 query among
10,000 documents, you must run 10,000 full forward passes *at query time*. The original SBERT paper points out
that finding the most similar pair among 10,000 sentences this way takes about 65 hours on a V100 GPU —
completely unworkable for real-time search.

---

## 3. What SBERT actually does

SBERT modifies BERT with a **siamese / triplet network** structure to fix exactly this:

1. **Two (or three) copies of the same BERT model**, with **shared weights**, each independently encode
   one sentence into token embeddings.
2. A **pooling layer** (mean pooling over token embeddings works best per the paper; `[CLS]`-token pooling
   and max pooling are alternatives) reduces the token embeddings into one **fixed-size sentence vector**
   (e.g. 384 or 768 dimensions).
3. During **training**, SBERT is fine-tuned on sentence-pair tasks — natural language inference (NLI) data
   (pairs labeled entailment/contradiction/neutral) and STS data (pairs with human similarity scores) — using
   loss functions specifically designed to shape the *embedding space*, e.g.:
   - **Softmax/classification loss** on NLI pairs
   - **Triplet loss**: pull an anchor and a positive example together, push a negative example away
   - **Cosine similarity / MSE loss**: directly regress predicted cosine similarity toward a human-labeled score
   - **MultipleNegativesRankingLoss** (used heavily in modern SBERT models): treat every other item in a
     batch as a negative, which is very sample-efficient and is the workhorse loss for today's general-purpose
     embedding models.

The crucial change: **the training objective directly optimizes for "cosine similarity should reflect semantic
similarity."** That's not a side effect — it's the explicit target. This is why SBERT vectors are usable for
search and clustering, while raw BERT vectors aren't.

Because of the *siamese* (weight-shared, independent encoding) structure, at **inference time** you do NOT
need to run the model on (query, document) pairs jointly. You encode every document **once, offline**, store
the vectors, and at query time you only need to encode the (single) query and do a fast vector comparison.
The same 10,000-sentence comparison that took 65 hours with a cross-encoder takes **about 5 seconds** with
SBERT (encoding) plus milliseconds for the cosine similarity computation, per the original paper's benchmark.

```
Cross-encoder (accurate, slow):     [CLS] query [SEP] doc [SEP]  -> BERT -> similarity score
                                     (must rerun BERT for every query-doc pair)

Bi-encoder / SBERT (fast, scalable): query -> BERT -> pooling -> vector_q   ┐
                                     doc   -> BERT -> pooling -> vector_d   ┘ -> cosine(vector_q, vector_d)
                                     (doc vectors precomputed once, reused for every query)
```

This query/document architecture — encode once, compare cheaply — is called a **bi-encoder**, and it's the
standard architecture behind essentially every modern semantic/vector search system (and the retrieval stage
of RAG pipelines).

---

## 4. The lexical baselines we'll compare against

**TF-IDF (Term Frequency–Inverse Document Frequency)**
- Represents each document as a sparse vector over the vocabulary.
- *Term Frequency*: how often a word appears in a document.
- *Inverse Document Frequency*: down-weights words that appear in many documents (e.g. "the") and
  up-weights words that are rare and therefore more informative.
- Similarity is typically cosine similarity between these sparse vectors.

**BM25 (Best Match 25)**
- The dominant ranking function in classical IR (it's what Elasticsearch/Lucene use by default).
- Improves on raw TF-IDF with two key ideas:
  - **Term frequency saturation**: the 10th occurrence of a word in a document shouldn't matter nearly
    as much as the 2nd occurrence — BM25 uses a saturating function instead of a linear count.
  - **Document length normalization**: longer documents naturally contain more word occurrences by chance,
    so BM25 normalizes by document length relative to the average.
- BM25 score for query terms $q_1 \dots q_n$ against document $D$:

$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

  where $f(q_i, D)$ is the term frequency of $q_i$ in $D$, $|D|$ is the document length, `avgdl` is the
  average document length across the corpus, and $k_1$ (~1.2–2.0) and $b$ (~0.75) are tuning constants.

Both methods only match **exact tokens** (after light normalization like lowercasing/stemming) — they have
no notion of synonyms or paraphrase, which is exactly the gap SBERT fills.

---

## 5. What we'll build in this notebook

1. A small but realistic multi-topic corpus (so semantic vs. lexical differences become obvious)
2. **TF-IDF + cosine** search engine
3. **BM25** search engine
4. **SBERT embeddings + cosine** semantic search engine (with a swappable FAISS index for scale)
5. Hand-labeled relevance judgments for a handful of queries (including paraphrase/synonym queries designed
   to break lexical search)
6. A benchmark: **Precision@k, Mean Reciprocal Rank (MRR), and latency** for all three methods
7. Discussion of failure modes and when to combine methods (hybrid search)


## 6. Setup

Install the libraries we need. `sentence-transformers` pulls in `torch` and `transformers`;
`rank_bm25` is a tiny pure-Python BM25 implementation; `scikit-learn` gives us TF-IDF and cosine similarity
for free. `faiss-cpu` is optional — only needed for the "scaling to millions of vectors" section near the end.


In [ ]:
!pip install -q sentence-transformers rank_bm25 scikit-learn faiss-cpu

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, util

import warnings
warnings.filterwarnings("ignore")

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_colwidth", 100)

## 7. Build the corpus

A deliberately mixed corpus across a few topics (tech support, cooking, finance, health, travel), with
some documents phrased very differently from how a user might query them. This is what will let us actually
*see* the semantic-vs-lexical gap rather than just trusting the theory.


In [ ]:
corpus = [
    # --- Tech support (password/login) ---
    "How do I reset my password if I forgot it?",
    "Steps to recover your account after losing access to your email.",
    "Two-factor authentication setup guide for your account.",
    "Our servers experienced downtime due to a database connection failure.",
    "Troubleshooting steps when your laptop will not turn on.",

    # --- Cooking ---
    "A simple recipe for baking sourdough bread at home.",
    "How to make a rich tomato pasta sauce from scratch.",
    "Tips for grilling steak to medium-rare perfection.",
    "The science behind why bread dough needs to rest before baking.",
    "Best practices for storing fresh vegetables to keep them from spoiling.",

    # --- Finance ---
    "Understanding compound interest and how it grows your savings over time.",
    "A beginner's guide to investing in low-cost index funds.",
    "What is inflation and how does it affect purchasing power?",
    "How to build an emergency fund before investing in the stock market.",
    "The difference between a Roth IRA and a traditional 401(k).",

    # --- Health & fitness ---
    "Why getting enough sleep is critical for muscle recovery after exercise.",
    "A beginner's guide to running your first 5K race.",
    "How drinking enough water supports overall metabolic health.",
    "Stretching routines to prevent injury before a workout.",
    "The benefits of strength training for long-term bone density.",

    # --- Travel ---
    "Best time of year to visit Japan for cherry blossom season.",
    "Tips for packing light for a two-week backpacking trip in Europe.",
    "How to find cheap flights by booking several weeks in advance.",
    "A guide to navigating public transportation in a new city.",
    "What to know before renting a car in a foreign country.",
]

print(f"Corpus size: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"[{i:2d}] {doc}")

## 8. Baseline 1 — TF-IDF + cosine similarity

`TfidfVectorizer` builds the sparse term-document matrix; cosine similarity between the query's TF-IDF
vector and every document's TF-IDF vector gives us a ranking.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)   # shape: (n_docs, vocab_size), sparse

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

In [ ]:
def tfidf_search(query, top_k=5):
    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    ranked_idx = np.argsort(-scores)[:top_k]
    return [(idx, scores[idx]) for idx in ranked_idx]


def show_results(results, label):
    print(f"--- {label} ---")
    for rank, (idx, score) in enumerate(results, start=1):
        print(f"{rank}. ({score:.4f}) [{idx:2d}] {corpus[idx]}")
    print()


query = "I forgot my login credentials, how can I get back into my account?"
show_results(tfidf_search(query), "TF-IDF")

Notice the query never uses the words "password" or "reset" — it says *"forgot my login credentials"*
and *"get back into my account."* TF-IDF can only match on overlapping vocabulary (after stopword removal),
so depending on word overlap with the corpus, it may completely miss document `[0]` ("reset my password")
and instead surface something only loosely related, purely because of incidental shared words like "account."
This is the lexical gap in action — keep this query in mind for comparison once we run SBERT.

## 9. Baseline 2 — BM25

`rank_bm25`'s `BM25Okapi` expects pre-tokenized documents (a list of token lists). We'll use simple
lowercase whitespace tokenization — good enough for this demo; production systems usually add stemming
and better tokenization.


In [ ]:
def tokenize(text):
    return text.lower().replace(",", "").replace(".", "").replace("?", "").split()


tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def bm25_search(query, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked_idx = np.argsort(-scores)[:top_k]
    return [(idx, scores[idx]) for idx in ranked_idx]


show_results(bm25_search(query), "BM25")

BM25 has the same fundamental limitation as TF-IDF here — it's still exact-token matching, just with
better term-frequency saturation and length normalization. It will tend to do a bit better than TF-IDF on
longer, noisier corpora, but it cannot bridge a vocabulary gap that TF-IDF also can't bridge.

## 10. Semantic search with Sentence-BERT

We'll use `all-MiniLM-L6-v2` — a small, fast, widely-used SBERT model (6 transformer layers, 384-dim
output vectors). It's not the most accurate SBERT model available, but it's a great balance of speed and
quality for demos and many production use cases. (Swap in `all-mpnet-base-v2` for higher accuracy at the
cost of more compute, if needed.)

**Key idea**: encode the entire corpus **once** into vectors and cache them. At query time, encode only
the query and compare.


In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode the whole corpus once. normalize_embeddings=True means the vectors have unit length,
# so cosine similarity reduces to a plain dot product -- faster and numerically convenient.
corpus_embeddings = sbert_model.encode(
    corpus,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print("Embedding matrix shape:", corpus_embeddings.shape)  # (n_docs, 384)

In [ ]:
def sbert_search(query, top_k=5):
    query_embedding = sbert_model.encode(
        query, convert_to_tensor=True, normalize_embeddings=True
    )
    # util.cos_sim handles the cosine similarity computation (dot product, since normalized)
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    top_results = np.argsort(-cos_scores.cpu().numpy())[:top_k]
    return [(idx, float(cos_scores[idx])) for idx in top_results]


show_results(sbert_search(query), "SBERT")

Now run all three side by side on the same paraphrase-heavy query, to see the gap directly.

In [ ]:
queries_to_demo = [
    "I forgot my login credentials, how can I get back into my account?",
    "What should I eat to help my muscles heal after working out?",   # paraphrase of sleep/recovery doc
    "Is now a good time to put money into the stock market?",          # paraphrase of investing docs
]

for q in queries_to_demo:
    print("=" * 100)
    print(f"QUERY: {q}")
    print("=" * 100)
    show_results(tfidf_search(q, top_k=3), "TF-IDF")
    show_results(bm25_search(q, top_k=3), "BM25")
    show_results(sbert_search(q, top_k=3), "SBERT")

## 11. Benchmarking: Precision@k, MRR, and latency

To benchmark properly we need **ground-truth relevance labels** — for each query, which document indices
are actually relevant? We hand-label a query set below, deliberately mixing:

- **Lexical-friendly queries** (share vocabulary with the relevant doc) — baselines should do fine
- **Semantic/paraphrase queries** (no shared vocabulary, same meaning) — this is where SBERT should win

### Metrics used

- **Precision@k**: of the top-k results returned, what fraction are actually relevant?
- **Mean Reciprocal Rank (MRR)**: for each query, take $1 / \text{rank of the first relevant result}$
  (0 if no relevant result appears in top-k), then average across queries. This rewards placing the
  relevant result *near the top*, not just somewhere in top-k.


In [ ]:
# Each entry: (query, set of relevant document indices in `corpus`)
eval_set = [
    # Lexical-friendly (word overlap exists)
    ("how to bake bread at home", {5, 8}),
    ("tips for investing in index funds", {11}),
    ("best time to visit Japan", {20}),
    ("how to grill steak", {7}),

    # Semantic / paraphrase (little to no word overlap with the relevant doc)
    ("I forgot my login credentials, how can I get back into my account?", {0, 1}),
    ("what should I eat to help my muscles heal after working out?", {15}),
    ("is now a good time to put money into the stock market?", {11, 13}),
    ("my computer screen stays black when I press the power button", {4}),
    ("how can I avoid getting hurt before I start exercising?", {18}),
    ("what's the cheapest way to book a trip across Europe?", {21, 22}),
]

print(f"Evaluation queries: {len(eval_set)}")
for q, rel in eval_set:
    print(f"  - \"{q}\"  -> relevant: {sorted(rel)}")

In [ ]:
def precision_at_k(ranked_idx, relevant_set, k):
    top_k = ranked_idx[:k]
    if len(top_k) == 0:
        return 0.0
    hits = sum(1 for idx in top_k if idx in relevant_set)
    return hits / k


def reciprocal_rank(ranked_idx, relevant_set):
    for rank, idx in enumerate(ranked_idx, start=1):
        if idx in relevant_set:
            return 1.0 / rank
    return 0.0


def evaluate_method(search_fn, eval_set, k=5):
    precisions, rr_scores, latencies = [], [], []
    for query, relevant_set in eval_set:
        start = time.perf_counter()
        results = search_fn(query, top_k=k)
        latencies.append(time.perf_counter() - start)

        ranked_idx = [idx for idx, _ in results]
        precisions.append(precision_at_k(ranked_idx, relevant_set, k))
        rr_scores.append(reciprocal_rank(ranked_idx, relevant_set))

    return {
        "Precision@k": np.mean(precisions),
        "MRR": np.mean(rr_scores),
        "Avg latency (ms)": np.mean(latencies) * 1000,
    }


K = 5
results_table = pd.DataFrame({
    "TF-IDF": evaluate_method(tfidf_search, eval_set, k=K),
    "BM25":   evaluate_method(bm25_search, eval_set, k=K),
    "SBERT":  evaluate_method(sbert_search, eval_set, k=K),
}).T

results_table

**What to expect**: on this hand-built eval set, TF-IDF and BM25 should perform reasonably on the
lexical-friendly queries but score **poorly** on the semantic/paraphrase queries (often 0 precision and 0
reciprocal rank, since the relevant document may not even appear near the top). SBERT should score
consistently well across both query types, because it doesn't depend on shared vocabulary at all.

On **latency**, expect the opposite pattern at this tiny corpus scale: TF-IDF/BM25 are pure CPU array/dict
lookups and are extremely fast at this size; SBERT pays a one-time cost to run a neural network forward
pass on the query (and uses a GPU efficiently, but is doing more computation per query on CPU). This gap
*shrinks* in relative terms as the corpus grows, because the cosine-similarity comparison step (which both
methods must do over the *whole* corpus) starts to dominate, and because SBERT's per-document cost was
already paid upfront, offline, at indexing time.

Let's break results down per-query so the pattern is visible rather than hidden in an average.


In [ ]:
def per_query_breakdown(eval_set, k=5):
    rows = []
    for query, relevant_set in eval_set:
        for name, fn in [("TF-IDF", tfidf_search), ("BM25", bm25_search), ("SBERT", sbert_search)]:
            ranked_idx = [idx for idx, _ in fn(query, top_k=k)]
            rows.append({
                "query": query[:45] + ("..." if len(query) > 45 else ""),
                "method": name,
                "precision@k": precision_at_k(ranked_idx, relevant_set, k),
                "reciprocal_rank": round(reciprocal_rank(ranked_idx, relevant_set), 3),
            })
    return pd.DataFrame(rows)


breakdown_df = per_query_breakdown(eval_set, k=K)
breakdown_df.pivot(index="query", columns="method", values="reciprocal_rank")

## 12. Scaling beyond a toy corpus

Our demo corpus has 25 documents, so every method is instant and "latency" barely means anything. In
practice, semantic search is normally deployed over corpora ranging from thousands to hundreds of millions
of documents, and the way each method scales matters a lot:

- **TF-IDF / BM25** scale via **inverted indexes** — for each query term, only documents containing that
  term need to be scored at all. This is why classical search engines (Elasticsearch, Lucene, Solr) can be
  blazing fast even at huge scale: the work is proportional to how many documents contain the query's terms,
  not the full corpus size.
- **SBERT-based search**, in the naive form we wrote above, computes cosine similarity against **every**
  document vector — this is **brute-force exact nearest-neighbor search**, $O(n)$ per query. That's fine
  for thousands of documents but becomes slow for millions.

For large-scale semantic search, you'd swap the brute-force comparison for an **Approximate Nearest Neighbor
(ANN) index** — libraries like **FAISS** (Meta), **HNSWlib**, or managed vector databases (Pinecone, Qdrant,
Weaviate, Milvus) build index structures (e.g. HNSW graphs, IVF clusters) that find *approximate* nearest
neighbors in sub-linear time, trading a small amount of recall for a large speedup.

Here's the same SBERT search re-implemented with a FAISS index, which is the production-realistic pattern:


In [ ]:
import faiss

# Move embeddings to numpy (FAISS expects float32 numpy arrays)
corpus_embeddings_np = corpus_embeddings.cpu().numpy().astype("float32")
dim = corpus_embeddings_np.shape[1]

# Since embeddings are normalized, inner product == cosine similarity.
# IndexFlatIP = exact search via inner product (still brute-force, but FAISS's optimized version).
# For real scale, swap this for e.g. faiss.IndexHNSWFlat(dim, 32) for approximate search.
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(corpus_embeddings_np)

print(f"FAISS index built with {faiss_index.ntotal} vectors of dimension {dim}")


def sbert_faiss_search(query, top_k=5):
    query_embedding = sbert_model.encode(
        query, convert_to_tensor=False, normalize_embeddings=True
    ).astype("float32").reshape(1, -1)

    scores, indices = faiss_index.search(query_embedding, top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0])]


show_results(sbert_faiss_search(query), "SBERT + FAISS")

At 25 documents this returns identical results to the plain `util.cos_sim` version — it's the exact
same math (`IndexFlatIP` is exact, not approximate). The benefit only shows up at scale: swapping
`IndexFlatIP` for `IndexHNSWFlat` or `IndexIVFFlat` is what lets this same code pattern stay fast at millions
of vectors, at the cost of being approximate (occasionally missing the true nearest neighbor in exchange for
speed).

## 13. Summary: when to use what

| | TF-IDF | BM25 | SBERT |
|---|---|---|---|
| **Matches on** | Exact word overlap | Exact word overlap (with better TF saturation + length norm) | Semantic meaning |
| **Catches synonyms/paraphrase** | No | No | Yes |
| **Catches exact keyword/code/name matches** | Yes, very reliably | Yes, very reliably | Not reliable — embeddings can blur exact identifiers |
| **Index build cost** | Cheap, CPU only | Cheap, CPU only | Needs a neural network forward pass per document (GPU helps) |
| **Query-time cost at scale** | Fast via inverted index | Fast via inverted index | Needs ANN index (FAISS/HNSW) to stay fast at scale |
| **Interpretability** | High — you can see which words matched | High | Low — similarity score doesn't decompose into "why" |
| **Typical weakness** | Vocabulary mismatch | Vocabulary mismatch | Struggles with rare proper nouns, exact codes, negation subtleties |

**The practical takeaway**: in real production search and RAG systems, the two approaches are usually
**combined**, not chosen between — this is called **hybrid search**:

1. Run BM25 and SBERT search in parallel.
2. Merge their ranked lists with a fusion method (e.g. **Reciprocal Rank Fusion**, or a weighted score
   combination).
3. Optionally **rerank** the merged top-N candidates with a slower but more accurate **cross-encoder**
   (the architecture we ruled out for first-stage retrieval earlier, because now it only needs to score
   a small shortlist, not the whole corpus).

This retrieve-cheap-then-rerank-accurately pattern (sometimes called "retrieve and rerank") gets the
speed of bi-encoders/lexical search with much of the accuracy of slower cross-encoders, and is the
standard architecture behind most modern semantic search and RAG pipelines.

### Things to try next
- Swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` (slower, more accurate) and re-run the benchmark.
- Add more adversarial paraphrase queries to `eval_set` and see where SBERT itself starts to fail
  (e.g. queries needing exact numeric/date matching, or negation: "a hotel that does NOT allow pets").
- Implement Reciprocal Rank Fusion to combine BM25 + SBERT rankings into one hybrid ranking.
- Add a `CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")` reranking step on top of the SBERT shortlist.
